In [11]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

# initialize browser
driver = webdriver.Chrome()
driver.get("https://www.wipo.int/gii-ranking/en/rank")

wait = WebDriverWait(driver, 10)
element = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "w-full")))

html_source = driver.page_source

# save to soup for scraping
soup = BeautifulSoup(html_source, 'html.parser')

# save to file just in case
with open('page_source.html', 'w', encoding='utf-8') as f:
    f.write(html_source)
    
print("HTML saved to 'page_source.html'")

driver.quit()

HTML saved to 'page_source.html'


In [15]:
tables_div = soup.find(class_="grid sm:grid-cols-2 lg:grid-cols-3 gap-x-20 gap-y-10")

In [ ]:
from io import StringIO
import pandas as pd

tables = pd.read_html(StringIO(str(tables_div)))

In [36]:
df_tables = pd.concat(tables, ignore_index=True).sort_values("GII rank")
df_tables.to_csv("GII_rank")

In [80]:
df = pd.read_csv("GII_2011_2024_long_format.tsv", sep='\t')
df

,Country,Year,Rank,Score
0,AGO,2011,-1,-1.00
1,AGO,2012,135,22.20
2,AGO,2013,135,23.46
3,AGO,2014,135,23.82
4,AGO,2015,120,26.20
...,...,...,...,...
2081,ZWE,2020,120,19.97
2082,ZWE,2021,113,21.90
2083,ZWE,2022,107,18.10
2084,ZWE,2023,117,16.50


In [64]:
import country_converter as coco
import numpy as np

converted_country = coco.convert(names=df.Country, to="name")

In [81]:
df.rename({"Country":"CountryISO"}, axis=1, inplace=True)
df['CountryName'] = converted_country

In [82]:
df_country_iso = df.groupby(['CountryName', 'CountryISO'], as_index=False).count()[['CountryName', 'CountryISO']]
df_country_iso.head()

,CountryName,CountryISO
0,Albania,ALB
1,Algeria,DZA
2,Angola,AGO
3,Argentina,ARG
4,Armenia,ARM


In [ ]:
df_country_iso.to_csv('country_iso.csv', index=False)

---

In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel("Global_Cybersecurity_Threats_2015-2024___.xlsx")
df.head()

,Country,Year,Attack Type,Target Industry,Financial Loss (in Million $),Number of Affected Users,Attack Source,Security Vulnerability Type,Defense Mechanism Used,Incident Resolution Time (in Hours)
0,China,2019,Phishing,Education,80.53,773169,Hacker Group,Unpatched Software,VPN,63
1,China,2019,Ransomware,Retail,62.19,295961,Hacker Group,Unpatched Software,Firewall,71
2,India,2017,Man-in-the-Middle,IT,38.65,605895,Hacker Group,Weak Passwords,VPN,20
3,UK,2024,Ransomware,Telecommunications,41.44,659320,Nation-state,Social Engineering,AI-based Detection,7
4,Germany,2018,Man-in-the-Middle,IT,74.41,810682,Insider,Social Engineering,VPN,68


In [10]:
order = ['Attack Type', 'Defense Mechanism Used']
df_grouped = df.groupby(order).count()['Country']
results = {}
for attack in df['Attack Type'].unique():
    results[attack] = f'{df_grouped[attack].idxmax()} ({df_grouped[attack].max()})'
def_given_at = pd.Series(results).reset_index().rename({"index": "GivenAttack", 0: "TopDefense"}, axis=1)
def_given_at

,GivenAttack,TopDefense
0,Phishing,Encryption (121)
1,Ransomware,VPN (104)
2,Man-in-the-Middle,Firewall (106)
3,DDoS,VPN (120)
4,SQL Injection,Antivirus (113)
5,Malware,Antivirus (111)


In [11]:
def_given_at.to_csv('TopDefense_GivenAttack.csv', index=False)

In [12]:
order = ['Defense Mechanism Used', 'Attack Type']
df_grouped = df.groupby(order).count()['Country']
results = {}
for defense in df['Defense Mechanism Used'].unique():
    results[defense] = f'{df_grouped[defense].idxmax()} ({df_grouped[defense].max()})'
at_given_def = pd.Series(results).reset_index().rename({"index": "GivenDefense", 0: "TopAttack"}, axis=1)
at_given_def

,GivenDefense,TopAttack
0,VPN,DDoS (120)
1,Firewall,Man-in-the-Middle (106)
2,AI-based Detection,DDoS (113)
3,Antivirus,SQL Injection (113)
4,Encryption,Phishing (121)


In [13]:
at_given_def.to_csv('TopAttack_GivenDefense.csv', index=False)